In [ ]:
import cv2
import numpy as np
import os
import random
from skimage.feature import hog, local_binary_pattern
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# === Parameters ===
GRID = 8
LBP_RADIUS = 1
LBP_POINTS = 8 * LBP_RADIUS
PATCH_SIZE = (64, 64)
input_folder = r"C:\Users\Dell\Desktop\IIT\ds203\PROJECT\project_images\images_resized"

n_clusters = 18 # Total KMeans clusters


In [ ]:
def get_patches(img, grid=8):
    h, w = img.shape[:2]
    ph, pw = h // grid, w // grid
    patches = []
    for i in range(grid):
        for j in range(grid):
            patch = img[i*ph:(i+1)*ph, j*pw:(j+1)*pw]
            patches.append(patch)
    return patches

def extract_features(patch):
    patch_resized = cv2.resize(patch, PATCH_SIZE, interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(patch_resized, cv2.COLOR_BGR2GRAY)
    
    # --- HOG ---
    hog_feat = hog(gray, pixels_per_cell=(8,8), cells_per_block=(2,2), feature_vector=True)
    
    # --- LBP ---
    lbp = local_binary_pattern(gray, LBP_POINTS, LBP_RADIUS, method='uniform')
    lbp_hist,_ = np.histogram(lbp.ravel(), bins=np.arange(0, LBP_POINTS+3), range=(0, LBP_POINTS+2))
    lbp_hist = lbp_hist / (lbp_hist.sum() + 1e-6)
    
    # --- Edge Density ---
    edges = cv2.Canny(gray, 50, 150)
    edge_density = np.sum(edges > 0) / edges.size
    
    # --- Entropy ---
    hist_gray = cv2.calcHist([gray],[0],None,[256],[0,256])
    hist_gray = hist_gray / (np.sum(hist_gray) + 1e-6)
    entropy = -np.sum(hist_gray * np.log2(hist_gray + 1e-6))
    
    # --- Very light color info (HSV mean) ---
    hsv = cv2.cvtColor(patch_resized, cv2.COLOR_BGR2HSV)
    hsv_mean = np.mean(hsv.reshape(-1,3), axis=0) / [180, 255, 255]
    
    return np.concatenate([hog_feat, lbp_hist, [edge_density, entropy], hsv_mean])

# === Extract all features ===
all_features = []
image_paths = []

file_list = [f for f in os.listdir(input_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Found {len(file_list)} images... extracting features...")

for idx, filename in enumerate(file_list, 1):
    path = os.path.join(input_folder, filename)
    img = cv2.imread(path)
    if img is None or img.ndim != 3:
        continue
    patches = get_patches(img, GRID)
    feats = [extract_features(p) for p in patches]
    all_features.extend(feats)
    image_paths.append(path)
    if idx % 10 == 0:
        print(f"Processed {idx}/{len(file_list)} images")

all_features = np.array(all_features)
print("✅ Extracted features for", len(image_paths), "images")
print("Feature matrix shape:", all_features.shape)

# === Normalize ===
scaler = StandardScaler()
all_features_scaled = scaler.fit_transform(all_features)

# Optional: save to reuse later
np.savez("features_cached.npz", features=all_features_scaled, image_paths=image_paths)


In [ ]:
data = np.load("features_cached.npz", allow_pickle=True)
all_features_scaled = data["features"]
image_paths = list(data["image_paths"])
print("✅ Loaded cached features:", all_features_scaled.shape)


In [ ]:
pca = PCA(n_components=0.95, random_state=42)
all_features_pca = pca.fit_transform(all_features_scaled)

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
labels = kmeans.fit_predict(all_features_pca)

print("✅ KMeans complete. Found", n_clusters, "clusters.")


In [ ]:
os.makedirs("cluster_previews", exist_ok=True)
patches_per_img = GRID * GRID
num_samples = 20  # number of patches shown per cluster

for c in range(n_clusters):
    idxs = np.where(labels == c)[0]
    if len(idxs) == 0:
        continue
    
    chosen = random.sample(list(idxs), min(num_samples, len(idxs)))
    fig, axes = plt.subplots(4, 5, figsize=(10, 8))
    fig.suptitle(f"Cluster {c}", fontsize=16)
    
    for ax, idx in zip(axes.flat, chosen):
        img_idx = idx // patches_per_img
        patch_idx = idx % patches_per_img
        img = cv2.imread(image_paths[img_idx])
        h, w = img.shape[:2]
        ph, pw = h // GRID, w // GRID
        i, j = divmod(patch_idx, GRID)
        patch = img[i*ph:(i+1)*ph, j*pw:(j+1)*pw]
        ax.imshow(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))
        ax.axis("off")
    
    plt.tight_layout()
    plt.savefig(f"cluster_previews/cluster_{c}.png")
    plt.close(fig)

print("📁 Saved cluster preview images in folder: cluster_previews/")


In [ ]:
animal_clusters = [2,17,7,8,11]
def visualize_red_grid(img, labels_for_img, animal_clusters, grid=8, suppress_sky=True, suppress_dark=True, suppress_blue=True, suppress_green=True):
    """
    Draws grid overlay on image with animal cluster highlights (red boxes).
    Suppresses common background colors like sky, blue, black edges, and green vegetation.
    """

    h, w = img.shape[:2]
    ph, pw = h // grid, w // grid
    overlay = img.copy()

    for i in range(grid):
        for j in range(grid):
            idx = i * grid + j
            y1, y2 = i * ph, (i + 1) * ph
            x1, x2 = j * pw, (j + 1) * pw

            patch = img[y1:y2, x1:x2]
            hsv_patch = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
            mean_h, mean_s, mean_v = np.mean(hsv_patch.reshape(-1, 3), axis=0)

            # Default color and thickness
            color = (255, 255, 255)
            thickness = 1

            # --- Background suppression logic ---
            if suppress_sky and (mean_v > 200 and mean_s < 40):
                # bright + low saturation = sky/clouds
                color = (255, 255, 255)
                thickness = 1

            elif suppress_dark and (mean_v < 40 or (mean_v < 65 and mean_s < 30)):
                # very dark = borders, shadows
                color = (255, 255, 255)
                thickness = 1

            elif suppress_blue and (90 < mean_h < 140):
                # blue tones = sky or water
                color = (255, 255, 255)
                thickness = 1

            elif suppress_green and (35 < mean_h < 85 and mean_s > 70):
                # green tones (light and dark greens)
                color = (255, 255, 255)
                thickness = 1

            elif labels_for_img[idx] in animal_clusters:
                # Animal cluster region → red overlay + red box
                color = (0, 0, 255)
                thickness = 2
                overlay[y1:y2, x1:x2] = cv2.addWeighted(
                    overlay[y1:y2, x1:x2], 0.6,
                    np.full_like(overlay[y1:y2, x1:x2], color), 0.4, 0
                )

            # Draw final box
            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, thickness, lineType=cv2.LINE_8)

    return overlay


output_folder = "cluster_visuals"
os.makedirs(output_folder, exist_ok=True)
patches_per_img = GRID * GRID
sample_indices = np.random.choice(len(image_paths), size=min(30, len(image_paths)), replace=False)

for idx in sample_indices:
    img = cv2.imread(image_paths[idx])
    start, end = idx * patches_per_img, (idx + 1) * patches_per_img
    labels_for_img = labels[start:end]

    grid_img = visualize_red_grid(
        img,
        labels_for_img,
        animal_clusters,
        GRID,
        suppress_sky=True,
        suppress_dark=True,
        suppress_blue=True,
        suppress_green=True
    )

    out_path = os.path.join(output_folder, f"vis_{os.path.basename(image_paths[idx])}")
    cv2.imwrite(out_path, grid_img)
    print("💾 Saved:", out_path)

    plt.figure(figsize=(6, 4))
    plt.imshow(cv2.cvtColor(grid_img, cv2.COLOR_BGR2RGB))
    plt.title(os.path.basename(image_paths[idx]))
    plt.axis("off")
    plt.show()

